# BantAI — Retraining run

**Sprint 4 · Track B (AI/ML) · WBS 4.3.5**

Runs one full cycle of the automated retraining pipeline on a Colab GPU:

```
snapshot  ->  fine-tune  ->  score both models  ->  promotion gate
```

and produces a `decision.json` recording whether the new checkpoint is
actually better than the deployed one.

### This is not the Sprint 2 notebook

`BantAI_Finetune_Colab.ipynb` trains a model from scratch on the labeled
dataset. This one runs `scripts/retrain.py`, which assembles a snapshot
(labeled dataset **+** admin-validated user reports), fine-tunes on that, and
then compares the result against the currently deployed checkpoint using
McNemar's test and an F1 floor.

It therefore needs **two** things you did not need in Sprint 2:

| | Where it comes from |
|---|---|
| `bantai_retrain_package.zip` | `cd ai && python colab/build_retrain_package.py` |
| The **current** model, as a baseline | `MyDrive/bantai/bantai_model.zip` — already there from the Sprint 2 run |

Without the baseline the gate has nothing to compare against: the pipeline
trains a candidate, writes `skipped_reason`, and refuses to invent a verdict.
That is a run that looks like it worked and proves nothing.

### Before you start
1. **Runtime → Change runtime type → T4 GPU.**
2. Rebuild the package if any code or data changed since you last did.

Expected runtime on a free T4: **~30-40 minutes** — the fine-tune, plus the
gate scoring both checkpoints over the validation split.

## 1. Confirm a GPU is attached

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU attached. Go to Runtime > Change runtime type > T4 GPU, '
        'then re-run this cell. (A CPU fine-tune of xlm-roberta-base over '
        '~16.8k rows takes many hours.)'
    )

print('GPU :', torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Install dependencies

Same stack as the Sprint 2 notebook. `transformers>=4.46` is required — that is
the release that renamed `Trainer(tokenizer=...)` to `processing_class`.

In [ ]:
%pip install -q 'transformers>=4.46' 'datasets>=2.19' accelerate sentencepiece

import datasets
import transformers

print('transformers', transformers.__version__)
print('datasets    ', datasets.__version__)

## 3. Upload the package

Run the cell, then pick **`bantai_retrain_package.zip`** (from `ai/colab/`).

Colab does not overwrite an existing file — a second upload of the same name
arrives as `bantai_retrain_package (1).zip` — so any earlier copy is deleted
first and the filename the picker actually returns is what gets unpacked.
Otherwise a re-run silently retrains from the stale package.

In [ ]:
import glob
import os

from google.colab import files

for stale in glob.glob('/content/bantai_retrain_package*.zip'):
    os.remove(stale)
    print('removed stale upload:', stale)

uploaded = files.upload()
PACKAGE = '/content/' + list(uploaded)[0]
print('\nwill unpack:', PACKAGE)

## 4. Unpack

In [ ]:
import shutil
import zipfile

shutil.rmtree('/content/bantai_ai', ignore_errors=True)
with zipfile.ZipFile(PACKAGE) as z:
    z.extractall('/content/bantai_ai')
%cd /content/bantai_ai

import sys

sys.path.insert(0, '.')

print()
!ls datasets/labeled retraining scripts

## 5. Restore the baseline checkpoint from Drive

The promotion gate compares the candidate against **the model currently
deployed**, so that model has to be present at `models/xlm-roberta-smishing/`.

This pulls it from `MyDrive/bantai/bantai_model.zip` — what the Sprint 2
notebook saved in its step 7. The zip already contains the
`models/xlm-roberta-smishing/` path, so it unpacks straight into place.

If this cell fails, do not just skip it. A missing baseline does not stop the
run; it produces a trained candidate with no verdict, which is the one
outcome that is easy to mistake for success.

In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')

BASELINE_ZIP = '/content/drive/MyDrive/bantai/bantai_model.zip'
if not os.path.isfile(BASELINE_ZIP):
    raise SystemExit(
        f'No baseline at {BASELINE_ZIP}.\n'
        'That file is written by BantAI_Finetune_Colab.ipynb step 7. Either run '
        'that notebook first, or upload your local ai/models/xlm-roberta-smishing/ '
        'to Drive as bantai_model.zip.'
    )

!unzip -qo '{BASELINE_ZIP}' -d /content/bantai_ai

BASELINE_DIR = 'models/xlm-roberta-smishing'
config = os.path.join(BASELINE_DIR, 'config.json')
if not os.path.isfile(config):
    raise SystemExit(
        f'Unzipped, but {config} is missing. The gate needs a real checkpoint '
        'directory here, not just any folder.'
    )

# Verify this is the checkpoint the service actually runs, not merely *a*
# checkpoint. A baseline that differs from the deployed model would make the
# gate compare against something nobody is using -- and the verdict would look
# entirely valid while meaning nothing. Recorded from the local
# ai/models/xlm-roberta-smishing/ on 2026-08-17.
import hashlib

EXPECTED_SHA = 'cd46599c7beea1dd82241b8f72f61c572de3fa92b1f56f75bfbe1769a86d7f3f'
EXPECTED_BYTES = 1112208084

weights = os.path.join(BASELINE_DIR, 'model.safetensors')
digest = hashlib.sha256()
with open(weights, 'rb') as handle:
    for chunk in iter(lambda: handle.read(1 << 22), b''):
        digest.update(chunk)
actual = digest.hexdigest()

print('size  :', os.path.getsize(weights), ' expected:', EXPECTED_BYTES)
print('sha256:', actual)

if actual != EXPECTED_SHA:
    print()
    print('BASELINE MISMATCH - this Drive copy is not the model your service runs.')
    print('Do not continue. The gate would compare the candidate against an')
    print('undeployed checkpoint and produce a verdict that means nothing.')
    print()
    print('Fix: run make-baseline-zip.ps1 locally, re-upload to Drive, re-run.')
    raise SystemExit('baseline mismatch')

print()
print('baseline verified - this is the deployed model.')

## 6. Check what the snapshot will consume

The whole point of retraining is the **validated user reports** — confirmed
mistakes, the only signal that says where the deployed model is actually
wrong. Everything else is just refreshing the model on data it already saw.

Colab cannot reach the backend on your laptop, so reports travel as a CSV
export produced by:

```bash
cd ai && python scripts/retrain.py --export-reports datasets/reports/validated.csv
```

and then picked up by `colab/build_retrain_package.py` into the zip. If the
next cell prints zero reports, that step was skipped — the run is still valid,
it just is not a correction-driven retrain, and the manifest will say so.

In [ ]:
import glob
import os

REPORTS_DIR = 'datasets/reports'
exports = sorted(glob.glob(os.path.join(REPORTS_DIR, '*.csv')) + glob.glob(os.path.join(REPORTS_DIR, '*.jsonl')))

if exports:
    from retraining.reports import FileReportSource

    reports = list(FileReportSource(REPORTS_DIR).fetch())
    print(f'{len(reports)} validated report(s) from: {", ".join(os.path.basename(e) for e in exports)}')
    for r in reports[:5]:
        print(f'  [{r.label}] {r.text[:70]}')
    REPORT_ARGS = ['--reports-dir', REPORTS_DIR]
else:
    print('No report export in the package.')
    print('This run will refresh the model on the labeled dataset alone,')
    print('and the manifest will record that no report store was consulted.')
    REPORT_ARGS = []

print('\nretrain.py args:', REPORT_ARGS or '(none)')

## 7. Run the retraining cycle

Everything happens inside `scripts/retrain.py`:

1. **Snapshot** — labeled dataset + validated reports, de-duplicated on
   *masked* text, with report labels winning on collision. Reports are never
   sampled away.
2. **Fine-tune** — `training/train.py` unchanged, pointed at the snapshot.
   Keeping one training implementation is what makes the candidate comparable
   to the incumbent.
3. **Gate** — both checkpoints score *the same rows in the same order*
   (McNemar's test depends on the pairing), then the F1 floor applies.

Nothing here swaps the live model. It writes a verdict and stops.

In [ ]:
import subprocess
import sys

command = [sys.executable, 'scripts/retrain.py', *REPORT_ARGS]
print('$', ' '.join(command), '\n', flush=True)

process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise SystemExit(f'retrain.py exited {process.returncode} -- see the output above.')

## 8. Read the verdict

`decision.json` carries every number the gate used, not just the outcome —
this is what gets read back when someone asks *why* a checkpoint was or was not
promoted.

- `promote: true` — the candidate beat the incumbent on macro-F1 and cleared
  McNemar. It is still **not** deployed; that is a deliberate separate step.
- `promote: false` — read `reason`. A rejection is a working gate, not a
  failed run.
- `skipped_reason` set — no baseline was present. Go back to step 5.

In [ ]:
import json
import os

runs = sorted(os.listdir('models/retraining_runs'))
RUN_DIR = os.path.join('models/retraining_runs', runs[-1])
print('run:', RUN_DIR, '\n')

for name in ('manifest.json', 'decision.json'):
    path = os.path.join(RUN_DIR, name)
    print('=' * 60)
    print(name)
    print('=' * 60)
    if os.path.isfile(path):
        with open(path, encoding='utf-8') as handle:
            print(json.dumps(json.load(handle), indent=2, sort_keys=True))
    else:
        print('(not written -- the run stopped before this stage)')
    print()

## 9. Save the run back to Drive

**Colab deletes everything when the session ends.** The candidate checkpoint is
~1.1 GB, so Drive is the reliable route.

The whole run directory is saved, not just the weights: `manifest.json` says
what went into the snapshot, `decision.json` says what the gate concluded, and
`snapshot/snapshot.csv` is the exact training input. A checkpoint without those
three cannot be defended later.

In [ ]:
import os

ARCHIVE = '/content/bantai_retrain_run.zip'
!rm -f {ARCHIVE}
!zip -qr {ARCHIVE} {RUN_DIR}
!ls -lh {ARCHIVE}

!mkdir -p '/content/drive/MyDrive/bantai/retraining_runs'
!cp {ARCHIVE} '/content/drive/MyDrive/bantai/retraining_runs/'
print('\nSaved to Drive: MyDrive/bantai/retraining_runs/bantai_retrain_run.zip')

## 10. Back on your laptop

1. Copy the contents of `manifest.json` and `decision.json` (printed in step 8)
   into a new file under **`ai/evaluation/`**, named
   `retraining_run_<date>.json`. That is where this project keeps measurement
   artifacts — see `embedding_centering_*.json` beside it.

   Not under `ai/models/retraining_runs/`: `ai/models/*/` is git-ignored, so
   the evidence would silently never be committed.

2. Leave the weights on Drive. They are ~1.1 GB and fully regenerable from the
   manifest — same dataset, same seed, same training code — so there is no
   reason to pull them down unless you are actually promoting.

3. Report the verdict so WBS 4.3.5 can be closed out.

### If the gate said promote

Promotion is not automatic, and it is not just a file copy:

```
1. Point the live model at the candidate directory
2. Re-run scripts/embed_dataset.py  AND  scripts/cluster_campaigns.py
```

Campaign centroids are tied to the checkpoint that produced them — they live in
that model's embedding space and are meaningless against a different one. A
promoted model with stale centroids silently breaks campaign matching. See
`ai/RETRAINING.md` § Rollback.